# TriageDesk: is this ticket a known issue?
**Use case (from the use-case sheet):** when a ticket arrives, search past Freshdesk tickets, Jira bugs
and KB articles, decide whether it is a known issue, a known bug or something new, and draft a grounded reply. It also
covers building a custom MCP server for systems that have no ready-made connector.

**What you will practise:** PII masking and prompt-injection screening, query rewriting, tool fan-out and link following,
grounded structured decisions, deterministic guardrails over model output, and a custom **MCP server** consumed from Python.

Cells marked **LIVE** call OpenAI; everything else runs offline.

## 0. Setup
Runs in Google Colab or locally. In Colab the first cell clones the repo and installs this project's requirements; locally, start Jupyter from the project folder.

In [ ]:
import os, sys, subprocess
from pathlib import Path

PROJECT = "04-use-case-projects/triagedesk-support-triage"
if "google.colab" in sys.modules:
    if not Path("genai-agentic-ai-training").exists():
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/tarang16/genai-agentic-ai-training.git"], check=True)
    os.chdir(f"genai-agentic-ai-training/{PROJECT}")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif Path(PROJECT).exists():  # started from the repo root
    os.chdir(PROJECT)
sys.path.insert(0, os.getcwd())
print("Working in", os.getcwd())

In [ ]:
# LIVE: only needed for the cells marked LIVE. Never paste a key into a cell; getpass keeps it out of the file.
from getpass import getpass
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("OpenAI API key: ")

## 1. The data
20 resolved tickets (Freshdesk CSV export shape), 5 Jira issues, 6 KB articles, and 6 new tickets to triage, including a safety case, a prompt-injection attempt and a genuinely new problem.

In [ ]:
import json, pandas as pd
from triagedesk.config import TICKETS_CSV, INCOMING
display(pd.read_csv(TICKETS_CSV).head(6))
new = [json.loads(l) for l in INCOMING.read_text().splitlines()]
pd.DataFrame(new)[["ticket_id", "product", "subject"]]

## 2. Guardrails before the model

In [ ]:
from triagedesk.guardrails import mask_pii, injection_suspected, safety_risk
print(mask_pii(new[0]["description"]))
for t in new:
    print(t["ticket_id"], "injection" if injection_suspected(t["description"]) else "", "SAFETY" if safety_risk(t["description"]) else "")

## 3. The tools
The same four functions back the LangGraph agent *and* the MCP server. Note how a past ticket links to `DISP-430`: the graph follows those links so the Jira issue becomes evidence even if search misses it.

In [ ]:
from triagedesk.store import SupportKnowledge
store = SupportKnowledge()
for h in store.search_past_tickets("monitor switches itself off every few minutes firmware 1.0.7", product="VX2780"):
    print(h["id"], h["score"], h["links"], h["title"])
print(store.get("DISP-430")["text"])

## 4. Your own MCP server
`mcp_server.py` wraps the tools with the official MCP Python SDK (v2: `MCPServer`). Here we connect to it in-process; `mcp_client_demo.py` does the same over stdio, exactly as a desktop MCP client would.

In [ ]:
import mcp_server
from mcp import Client

async def demo():
    async with Client(mcp_server.server) as client:
        print([t.name for t in (await client.list_tools()).tools])
        res = await client.call_tool("search_jira", {"query": "touch not responding after update", "product": "TD2455"})
        print(res.content[0].text[:400])
await demo()

To use it from any MCP client, add this to the client's MCP configuration file (for example `mcp.json`):
```json
{"mcpServers": {"triagedesk": {"command": "python", "args": ["/absolute/path/to/mcp_server.py"]}}}
```

## 5. The triage graph

In [ ]:
# LIVE
from triagedesk import create_app, render
app = create_app()
print(app.get_graph().draw_mermaid())
for t in new:
    state = app.invoke({"ticket": t})
    print(render(state)); print("-" * 80)

## 6. Evaluate
`python evaluate.py --offline` checks evidence recall without a key; `python evaluate.py` adds verdict, reference and escalation accuracy.

In [ ]:
import subprocess, sys
print(subprocess.run([sys.executable, "evaluate.py", "--offline"], capture_output=True, text=True).stdout)

## 7. Take it further
1. Replace `store.py` loaders with real calls: Freshdesk `GET /api/v2/search/tickets`, Jira `GET /rest/api/3/search/jql`.
2. Add embeddings next to BM25 and measure evidence recall on paraphrased tickets.
3. Expose `triage_ticket` as an MCP tool so an MCP client can run the whole pipeline.
4. Add a LangGraph `interrupt()` before any reply is sent when `auto_send_ok` is false.